In this notebook we'll explore different document loaders available through `langchain`.

## PyPDF loader

This is suited to simple, clean PDFs.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

/Users/ashapatel/Documents/projects/rag_cc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/insurance-product-information-document.pdf"

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/policy-wording.pdf"

In [18]:
loader1 = PyPDFLoader(doc1_path)
document1 = loader1.load()

In [19]:
loader2 = PyPDFLoader(doc2_path)
document2 = loader2.load()

### Explore the extracted page content

In [16]:
for page_num, page in enumerate(document1):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

=== Page 1 ===
Admiral Insurance
Insurance Product Information Document
Company: EUI Limited                 Product: Platinum Travel Insurance (Single Trip and Annual Multi Trip cover)
EUI Limited is registered in the UK and is authorised and regulated by the Financial Conduct Authority, Financial Services Register 
reference number: 309378
This document summarises the key features of your insurance policy. It is not tailored to individual needs and so may not provide all the information relevant to your cover 
requirements. Complete pre-contractual and contractual information is provided in other documents.  
What is this type of insurance?
This policy provides cover against specific events while travelling away from home, as summarised below. You may also include optional policy upgrades for an  
additional premium.
What is insured?
  Emergency medical costs and repatriation: up to £20,000,000 
for emergency medical costs including, up to £400 for 
emergency dental pain relief, up t

The first page in document1 contains 2 columns each containing bullet points. This document loader correctly extracted each bullet point, reading down the left-hand column then the right-hand column.

In [20]:
for page_num, page in enumerate(document2):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

=== Page 1 ===
Guide to your Travel  
Insurance cover
Please call us immediately if you need hospital treatment, your medical
expenses are likely to be more than £500, or you need to cut your trip short.
  +44 (0)292 010 7777

=== Page 2 ===
Your Cover with Admiral 
ii
Welcome to Admiral 
This policy booklet provides all the details you need to know about your travel insurance with us, EUI Limited. Your 
policy is underwritten by the authorised insurer (Admiral Insurance (Gibraltar) Limited). They have agreed to cover 
you under the terms, conditions, limitations and exclusions described in this policy booklet. 
Suitability of cover 
This policy is suitable for customers who want to insure against specific events related to travelling away from your 
home. There are three levels of cover.
 y Admiral
 y Gold
 y Platinum 
Each level has different features and benefits. There is further information in the Insurance Product Information 
Document for each level of cover.
Reciprocal health a

The second document looks good too. The 4-column table in the appendix was extracted row-by-row.

One issue I noticed is that some bullet points have been extracted as the letter 'Y'.

### Cleaning page content

Looking at the page content, there's a lot of whitespace e.g between section headings and the text in the given section. We could remove this whitespace so that the section heading and text are more likely to appear in the same chunk, which helps preserve context and also reduces token usage.

Let's write a function to clean the page content:
- Collapse whitespace, i.e multiple spaces or tabs back-to-back
- Wherever 3 or more new lines occur consecutively, replace them with 2 new lines

The new line processing is informed by the fact that LangChain's `RecursiveCharacterTextSplitter` -- which we'll use for chunking -- looks for `\n\n` as the plain-text representation of a paragraph break.

In [5]:
import re


def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


assert clean_text(" Hello   World ") == "Hello World"
assert clean_text("Hello\n\n\nWorld") == "Hello\n\nWorld"

Let's test this cleaning function on a page from our first document.

In [6]:
doc1_page_content = documents[0].page_content
print(doc1_page_content)

Admiral Insurance
Insurance Product Information Document
Company: EUI Limited                 Product: Platinum Travel Insurance (Single Trip and Annual Multi Trip cover)
EUI Limited is registered in the UK and is authorised and regulated by the Financial Conduct Authority, Financial Services Register 
reference number: 309378
This document summarises the key features of your insurance policy. It is not tailored to individual needs and so may not provide all the information relevant to your cover 
requirements. Complete pre-contractual and contractual information is provided in other documents.  
What is this type of insurance?
This policy provides cover against specific events while travelling away from home, as summarised below. You may also include optional policy upgrades for an  
additional premium.
What is insured?
  Emergency medical costs and repatriation: up to £20,000,000 
for emergency medical costs including, up to £400 for 
emergency dental pain relief, up to £1,000 for as

In [7]:
cleaned_content = clean_text(doc1_page_content)
print(cleaned_content)

Admiral Insurance
Insurance Product Information Document
Company: EUI Limited Product: Platinum Travel Insurance (Single Trip and Annual Multi Trip cover)
EUI Limited is registered in the UK and is authorised and regulated by the Financial Conduct Authority, Financial Services Register 
reference number: 309378
This document summarises the key features of your insurance policy. It is not tailored to individual needs and so may not provide all the information relevant to your cover 
requirements. Complete pre-contractual and contractual information is provided in other documents. 
What is this type of insurance?
This policy provides cover against specific events while travelling away from home, as summarised below. You may also include optional policy upgrades for an 
additional premium.
What is insured?
 Emergency medical costs and repatriation: up to £20,000,000 
for emergency medical costs including, up to £400 for 
emergency dental pain relief, up to £1,000 for associated travel & 


Comparing this to the original page content before cleaning, we can see that the large sections of whitespace have been removed. 

Now let's test the cleaning function on some pages from our second document, which is much longer (84 pages) and has more variety in page layout.

In [8]:
loader = PyPDFLoader(doc2_path)
documents = loader.load()

doc2_page_content = documents[1].page_content
print(doc2_page_content)

Your Cover with Admiral 
ii
Welcome to Admiral 
This policy booklet provides all the details you need to know about your travel insurance with us, EUI Limited. Your 
policy is underwritten by the authorised insurer (Admiral Insurance (Gibraltar) Limited). They have agreed to cover 
you under the terms, conditions, limitations and exclusions described in this policy booklet. 
Suitability of cover 
This policy is suitable for customers who want to insure against specific events related to travelling away from your 
home. There are three levels of cover.
 y Admiral
 y Gold
 y Platinum 
Each level has different features and benefits. There is further information in the Insurance Product Information 
Document for each level of cover.
Reciprocal health agreements
If you are travelling to a country in the EU, or to Switzerland, Norway, Iceland or Liechtenstein, you should apply for a 
Global Health Insurance Card (GHIC) on the website at www.gov.uk, unless you have a valid European Health Ins

In [9]:
cleaned_content = clean_text(doc2_page_content)
print(cleaned_content)

Your Cover with Admiral 
ii
Welcome to Admiral 
This policy booklet provides all the details you need to know about your travel insurance with us, EUI Limited. Your 
policy is underwritten by the authorised insurer (Admiral Insurance (Gibraltar) Limited). They have agreed to cover 
you under the terms, conditions, limitations and exclusions described in this policy booklet. 
Suitability of cover 
This policy is suitable for customers who want to insure against specific events related to travelling away from your 
home. There are three levels of cover.
 y Admiral
 y Gold
 y Platinum 
Each level has different features and benefits. There is further information in the Insurance Product Information 
Document for each level of cover.
Reciprocal health agreements
If you are travelling to a country in the EU, or to Switzerland, Norway, Iceland or Liechtenstein, you should apply for a 
Global Health Insurance Card (GHIC) on the website at www.gov.uk, unless you have a valid European Health Ins

### Explore the extracted metadata

In [10]:
for page_num, page in enumerate(documents):
    print(f"=== Page {page_num + 1} ===")
    print(page.metadata)
    print()

=== Page 1 ===
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-11-04T13:51:09+00:00', 'moddate': '2025-11-04T13:51:12+00:00', 'trapped': '/False', 'source': '/Users/ashapatel/Documents/projects/rag_cc/documents/policy-wording.pdf', 'total_pages': 84, 'page': 0, 'page_label': 'i'}

=== Page 2 ===
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-11-04T13:51:09+00:00', 'moddate': '2025-11-04T13:51:12+00:00', 'trapped': '/False', 'source': '/Users/ashapatel/Documents/projects/rag_cc/documents/policy-wording.pdf', 'total_pages': 84, 'page': 1, 'page_label': 'ii'}

=== Page 3 ===
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-11-04T13:51:09+00:00', 'moddate': '2025-11-04T13:51:12+00:00', 'trapped': '/False', 'source': '/Users/ashapatel/Documents/projects/rag_cc/documents/policy-wording.pdf', 'total_pages': 84, 'page':

Looking at the metadata, there are a few things that would be useful to persist to the chunking step:
- `source`: currently this gives the whole file path, but we're just interested in the file name
- `page`: this will enable citations referencing the specific page number on which the chunk appears in the original document

We're not interested in metadata such as `producer`, `creator` and `moddate` as they don't relate to the document's content.

## Directory loader

In [11]:
from langchain_community.document_loaders import DirectoryLoader

In [12]:
directory_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/"

loader = DirectoryLoader(
    directory_path, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
)
documents = loader.load()

print(f"Number of Documents: {len(documents)}")

for idx, document in enumerate(documents, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

100%|██████████| 2/2 [00:00<00:00,  2.58it/s]

Number of Documents: 86

Document 1
Content:
 Guide to your Travel  
Insurance cover
Please call us immediately if you need hospital treatment, your medical
expenses are likely to be more than £500, or you need to cut your trip short.
  +44 (0)292 010 7777
Metadata:
 {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-11-04T13:51:09+00:00', 'moddate': '2025-11-04T13:51:12+00:00', 'trapped': '/False', 'source': '/Users/ashapatel/Documents/projects/rag_cc/documents/policy-wording.pdf', 'total_pages': 84, 'page': 0, 'page_label': 'i'}

Document 2
Content:
 Your Cover with Admiral 
ii
Welcome to Admiral 
This policy booklet provides all the details you need to know about your travel insurance with us, EUI Limited. Your 
policy is underwritten by the authorised insurer (Admiral Insurance (Gibraltar) Limited). They have agreed to cover 
you under the terms, conditions, limitations and exclusions described in this policy booklet. 
Suitabi

This loads all documents in a given directory. This could be useful for my pipeline, but to start with I think I'll specify each document path separately.

# Classify page elements

In [2]:
from langchain_community.document_loaders import UnstructuredPDFLoader

/Users/ashapatel/Documents/projects/rag_cc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This loader is suited to PDFs with complex layouts, or scanned documents which require OCR to extract the text. It classifies page elements such as titles, tables, paragraphs etc.

In [ ]:
loader1 = UnstructuredPDFLoader(doc1_path)
documents1 = loader1.load()

print(f"Number of Documents: {len(documents1)}")

for idx, document in enumerate(documents1, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

No languages specified, defaulting to English.


Number of Documents: 1

Document 1
Content:
 Admiral Insurance Insurance Product Information Document

Company: EUI Limited Product: Platinum Travel Insurance (Single Trip and Annual Multi Trip cover) EUI Limited is registered in the UK and is authorised and regulated by the Financial Conduct Authority, Financial Services Register reference number: 309378

This document summarises the key features of your insurance policy. It is not tailored to individual needs and so may not provide all the information relevant to your cover requirements. Complete pre-contractual and contractual information is provided in other documents.

What is this type of insurance? This policy provides cover against specific events while travelling away from home, as summarised below. You may also include optional policy upgrades for an additional premium.

What is insured?

What is not insured?

Emergency medical costs and repatriation: up to £20,000,000 for emergency medical costs including, up to £400 for eme

In [4]:
loader2 = UnstructuredPDFLoader(doc2_path)
documents2 = loader2.load()

print(f"Number of Documents: {len(documents2)}")

for idx, document in enumerate(documents2, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

No languages specified, defaulting to English.


Number of Documents: 1

Document 1
Content:
 Guide to your Travel Insurance cover

Please call us immediately if you need hospital treatment, your medical expenses are likely to be more than £500, or you need to cut your trip short.

+44 (0)292 010 7777

ii

Welcome to Admiral

This policy booklet provides all the details you need to know about your travel insurance with us, EUI Limited. Your policy is underwritten by the authorised insurer (Admiral Insurance (Gibraltar) Limited). They have agreed to cover you under the terms, conditions, limitations and exclusions described in this policy booklet.

Suitability of cover

This policy is suitable for customers who want to insure against specific events related to travelling away from your home. There are three levels of cover.

y Admiral

y Gold

y Platinum

Each level has different features and benefits. There is further information in the Insurance Product Information Document for each level of cover.

Reciprocal health agreements

If 

For the first document, this loader mixes up the order of bullet points from the left-hand and right-hand columns, i.e doesn't read down the left-hand column first like the basic PyPDFLoader does.

For the second document, like PyPDFLoader this loader extracts some bullet points as the letter 'Y', but correctly reads the appendix table row by row.

Even if this loader did work better than simpler ones, we'd need to consider the extra processing time it takes.